In [6]:
import os
import sys
import gymnasium as gym
import numpy as np
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import MatchKickoffReset
from src.rl.reward_shapers import Stage2Reward
from src.rl.trainer import train_ppo_vectorized

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")


def make_sparring_env(
    red_players: int = 1,
    blue_players: int = 1,
    time_limit: float = 60.0,    # 60s matches
    score_limit: int = 0,        # 0 = Continuous play (no early reset on score)
    max_steps: int = 3600,       # 60s * 60 FPS = 3600 steps
):
    def _init():
        red_coord = TeamHeuristicCoordinator(team="red")
        blue_coord = TeamHeuristicCoordinator(team="blue")
        roster = []

        # 1. Red Team Slot 0: Primary RL Agent
        roster.append(PlayerSlot(
            team="red",
            stats=PlayerStats(name="RL_Agent", accel=3200.0)
        ))

        # 2. Additional Red Teammates (for N vs M)
        for i in range(1, red_players):
            roster.append(PlayerSlot(
                team="red",
                stats=PlayerStats(name=f"Red_Bot_{i}", accel=3000.0),
                controller=HeuristicBotController(red_coord)
            ))

        # 3. Blue Opponents
        for j in range(blue_players):
            roster.append(PlayerSlot(
                team="blue",
                stats=PlayerStats(name=f"Blue_Bot_{j + 1}", accel=3000.0),
                controller=HeuristicBotController(blue_coord)
            ))

        match_cfg = MatchConfig(
            mode=ClassicMatchMode(
                time_limit=time_limit,
                score_limit=score_limit,
                kickoff_timeout=6.0,
            ),
            roster=roster,
            time_limit=time_limit,
            score_limit=score_limit,
        )

        return HaxballGymEnv(
            match_config=match_cfg,
            reward_shaper=Stage2Reward(team="red"),
            reset_strategy=MatchKickoffReset(),
            max_steps=max_steps,
        )
    return _init


# --- Match Scaling Configuration ---
RED_COUNT = 1
BLUE_COUNT = 1
NUM_ENVS = 16
OBS_DIM = 80

# Training Environments: 60s continuous sparring sessions
train_envs = gym.vector.AsyncVectorEnv(
    [make_sparring_env(
        red_players=RED_COUNT,
        blue_players=BLUE_COUNT,
        time_limit=60.0,
        score_limit=0,
        max_steps=3600
    ) for _ in range(NUM_ENVS)]
)

# Evaluation Environment: Standard 60s match with 3-goal threshold
eval_env = make_sparring_env(
    red_players=RED_COUNT,
    blue_players=BLUE_COUNT,
    time_limit=60.0,
    score_limit=3,
    max_steps=3600
)()

# Model Initialization
model = ActorCritic(obs_dim=OBS_DIM).to(device)

# Weight Transfer from Stage 1 Checkpoint
stage1_ckpt = "models/stage1/best_model.pt"
if os.path.exists(stage1_ckpt):
    ckpt = torch.load(stage1_ckpt, map_location=device, weights_only=False)
    state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

    old_weight = state_dict.get("shared.0.weight", None)
    if old_weight is not None and old_weight.shape[1] == 68:
        print("🔄 Transplanting 68-dim weights into 80-dim architecture...")
        with torch.no_grad():
            model.shared[0].weight.data[:, :68] = old_weight
            for key in state_dict:
                if key != "shared.0.weight" and key in model.state_dict():
                    model.state_dict()[key].copy_(state_dict[key])
        print("✅ Transferred baseline motor skills from Stage 1!")
    elif old_weight is not None and old_weight.shape[1] == 80:
        model.load_state_dict(state_dict)
        print("✅ Loaded matching 80-dim checkpoint directly.")



⚡ Device: cuda
✅ Loaded matching 80-dim checkpoint directly.


In [7]:
# Vectorized Training Execution
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=100_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=100,
    save_dir="models/stage2",
    lr_initial=2.5e-4,
    lr_final=1e-5,
    ent_coef_initial=0.01,
    ent_coef_final=0.001,
)

train_envs.close()
eval_env.close()

🚀 Training (80 dims) | Benchmark every 100000 steps...

📊 [EVALUATION @ Step  102400] Scored:  24.0% (24/100) | Conceded:  85.0% (85/100) | Net: -61 | Touch: 100.0% | Avg Steps: 3600.0 | Mean Reward: -162.52
   ⭐ New verified best model saved: models/stage2/best_model.pt
      [Net: -61 | Scored: 24.0% | Reward: -162.52 | Speed: 3600.0 steps]

✅ Training completed. Final model saved to models/stage2/final_model.pt


# Testing

In [14]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load RL models
obs_dim = 80
stage2_model = ActorCritic(obs_dim).to(device)
stage2_model.load_state_dict(torch.load("models/stage1/best_model.pt", map_location=device))


# 2. Setup Team Coordinators
red_rl_controller = RLController(stage2_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(stage2_model, team="blue")



In [16]:
# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 60.0s per episode (5 Episodes)
   Episode 1: 4 goals
   Episode 2: 3 goals
   Episode 3: 2 goals
   Episode 4: 2 goals
   Episode 5: 1 goals
📊 Average Scoring Rate: 2.40 goals / 60.0s



2.4

In [13]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (1 Matches)
✅ Completed in 6.57s
🏆 Series Outcome (Wins): RED 0 | BLUE 1 | DRAWS 0
⚽ Avg Goals / Match:     RED 2.00 | BLUE 3.00



In [11]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: BLUE WINS! 🎉 (3 - 2)
Replay saved to: renders/arena/2026-08-22_00-55-55_match_1.html



In [12]:
from src.rl.benchmarker import render_solo_drill

device = torch.device("cpu")
model = ActorCritic(obs_dim=80).to(device)
model.load_state_dict(torch.load("models/stage2/best_model.pt", map_location=device, weights_only=False))

agent_slot = PlayerSlot("red", PlayerStats("RL_Agent"), RLController(model, team="red", device=device))

# Render 3 randomized episodes (20 seconds each)
replay_path = render_solo_drill(
    agent_slot=agent_slot,
    num_episodes=3,
    time_limit=20.0,
    save_path="renders/solo_drills",
)

🎬 Generating 3 Solo Drill Replays (20.0s each)...
   Episode 1 Finished: 0 Goals Scored
   Episode 2 Finished: 0 Goals Scored
   Episode 3 Finished: 0 Goals Scored
🏆 Overall: 0.00 Avg Goals / 20.0s
Replay saved to: renders/solo_drills/2026-08-22_00-55-59_solo_drill.html

